# Neural Encoder Interpretation

## Imports and Setup

In [1]:
!pip install lightning captum

In [2]:
!git clone https://github.com/ZareiShayan/tvsd-encode.git
%cd /kaggle/working/tvsd-encode

fatal: destination path 'tvsd-encode' already exists and is not an empty directory.
/kaggle/working/tvsd-encode


In [3]:
!git status
!git pull origin main

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	models/
	plot/
	src/__pycache__/__init__.cpython-312.pyc
	src/__pycache__/classes.cpython-312.pyc
	src/__pycache__/helper_functions.cpython-312.pyc
	src/__pycache__/plot_functions.cpython-312.pyc
	tvsd-encode/

nothing added to commit but untracked files present (use "git add" to track)
From https://github.com/ZareiShayan/tvsd-encode
 * branch            main       -> FETCH_HEAD
Already up to date.


In [4]:
from pathlib import Path
import shutil
import sys

PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import h5py
import numpy as np
import pandas as pd
import torch
from lightning.pytorch.utilities.combined_loader import CombinedLoader

from src.classes import *
from src.helper_functions import *
from src.plot_functions import *

SEED = 1
seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
import importlib
import src.classes as classes
import src.helper_functions as helper_functions
import src.plot_functions as plot_functions

importlib.reload(classes)
importlib.reload(helper_functions)
importlib.reload(plot_functions)

from src.classes import *
from src.helper_functions import *
from src.plot_functions import *

## Configuration

In [6]:
Conf = DotDict({
    "run_id": 1,
    "seed": SEED,
    "paths": {
        "data": "/kaggle/input/datasets/shayanzarei/things-encoder/data.mat",
        "metadata": "/kaggle/input/datasets/shayanzarei/things-encoder/data_metadata.csv",
    },
    "device": device,
    "data": {
        "bin_size": 0.01,
    },
    "training": {
        "batch_size": 64,
        "max_epoch": 500,
        "min_delta": 1e-5,
        "patience": 6,
    },
    "model_type": {
        "name": "cnn_latent",
        "cnn_latent": {
            "cnn_n_hidden": 8,
            "cnn_n_layers": 1,
            "n_latent": 32,
            "dropout": 0.2,
        },
    },
    "optimization": {
        "Adam": {
            "lr": 0.001,
            "weight_decay": 0,
        },
        "Reduce": {
            "factor": 0.5,
            "patience": 2,
            "min_lr": 1e-10,
        },
        "entropy_lambda": 0.0,
        "coverage_lambda": 0.0,
    },
})

## Data Preparation

In [7]:
with h5py.File(Conf.paths.data, "r") as f:
    g = f["data"]

    allmat = np.array(g["ALLMAT"]).T
    spikes = np.array(g["ALLMUA"]).transpose(1, 2, 0)
    images = np.array(g["IMAGES"]).transpose(0, 2, 1, 3)
    bin_times = np.array(g["tb"]).ravel()
    electrode_names = np.array(g["selectedElectrodes"]).ravel().astype(int)
    mapping = np.array(g["mapping"]).ravel().astype(int)


train_idx = allmat[:, 1].astype(int)
test_idx = allmat[:, 2].astype(int)
rep = allmat[:, 3].astype(int)
count = allmat[:, 4].astype(int)
day = allmat[:, 5].astype(int)

metadata = pd.read_csv(Conf.paths.metadata)
image_names = metadata["image_name"].str.rsplit(".", n=1).str[0].to_numpy()

electrode_roi = np.empty(len(electrode_names), dtype="<U2")
electrode_roi[electrode_names <= 512] = "V1"
electrode_roi[(electrode_names >= 513) & (electrode_names <= 832)] = "IT"
electrode_roi[electrode_names >= 833] = "V4"

In [8]:
day_values = np.unique(day)

allmat_list = []
spikes_list = []
images_list = []

train_idx_list = []
test_idx_list = []
rep_list = []
count_list = []
day_list = []

image_names_list = []

electrode_names_list = []
electrode_roi_list = []

bin_times_list = []
mapping_list = []

for day_value in day_values:
    day_mask = day == day_value

    allmat_list.append(allmat[day_mask])
    spikes_list.append(spikes[day_mask])
    images_list.append(images[day_mask])

    train_idx_list.append(train_idx[day_mask])
    test_idx_list.append(test_idx[day_mask])
    rep_list.append(rep[day_mask])
    count_list.append(count[day_mask])
    day_list.append(day[day_mask])

    image_names_list.append(image_names[day_mask])

    electrode_names_list.append(electrode_names.copy())
    electrode_roi_list.append(electrode_roi.copy())

    bin_times_list.append(bin_times.copy())
    mapping_list.append(mapping.copy())

del images, spikes, allmat

In [9]:
# trials_to_remove_list = []
# electrodes_to_remove_list = []

# for day_idx, day_value in enumerate(day_values):
#     trials_to_remove, electrodes_to_remove = plot_spike_summary(
#         spikes=spikes_list[day_idx],
#         file_name="spike-summary-before",
#         file_path=f"./plot/day_{day_value}/spikes",
#         metric="outlier_rate",
#         k=5.0,
#         electrode_threshold=0.02,
#         trial_threshold=0.02,
#         electrode_names=electrode_names_list[day_idx],
#     )

#     trials_to_remove_list.append(trials_to_remove)
#     electrodes_to_remove_list.append(electrodes_to_remove)

#     for electrode_idx in electrodes_to_remove:
#         plot_electrode_spikes(
#             electrode_spikes=spikes_list[day_idx][:, electrode_idx, :],
#             title=f"Day {day_value} - Electrode {electrode_names_list[day_idx][electrode_idx]}",
#             file_name=f"electrode-{electrode_names_list[day_idx][electrode_idx]}",
#             file_path=f"./plot/day_{day_value}/spikes/electrodes_to_remove",
#             bin_times=bin_times_list[day_idx],
#         )

In [10]:
electrodes_to_remove_list = [
    [173, 356, 463, 724, 732, 756, 768, 853, 864],
    [137, 264, 316, 461, 799, 820, 839, 853],
    [15, 474, 695, 750, 836],
    [],
]
trials_to_remove_list = [
    [],
    [],
    [],
    [],
]

keep_trials_mask_list = []
keep_electrodes_mask_list = []

n_trials_list = []
n_electrodes_list = []

for day_idx, day_value in enumerate(day_values):
    spikes_day = spikes_list[day_idx]
    images_day = images_list[day_idx]

    clean_trials_mask = np.ones(spikes_day.shape[0], dtype=bool)
    clean_trials_mask[trials_to_remove_list[day_idx]] = False

    spike_trials_mask_finite = np.isfinite(spikes_day).any(axis=(1, 2))
    image_trials_mask_finite = np.isfinite(images_day).any(axis=(1, 2, 3))
    finite_trials_mask = spike_trials_mask_finite & image_trials_mask_finite

    keep_trials_mask = clean_trials_mask & finite_trials_mask

    clean_electrodes_mask = ~np.isin(
        electrode_names_list[day_idx],
        electrodes_to_remove_list[day_idx],
    )

    nonnegative_electrodes_mask = (spikes_day >= 0).all(axis=(0, 2))

    keep_electrodes_mask = clean_electrodes_mask & nonnegative_electrodes_mask

    spikes_list[day_idx] = spikes_day[keep_trials_mask][:, keep_electrodes_mask, :]
    images_list[day_idx] = images_day[keep_trials_mask]
    allmat_list[day_idx] = allmat_list[day_idx][keep_trials_mask]

    train_idx_list[day_idx] = train_idx_list[day_idx][keep_trials_mask]
    test_idx_list[day_idx] = test_idx_list[day_idx][keep_trials_mask]
    rep_list[day_idx] = rep_list[day_idx][keep_trials_mask]
    count_list[day_idx] = count_list[day_idx][keep_trials_mask]
    day_list[day_idx] = day_list[day_idx][keep_trials_mask]
    image_names_list[day_idx] = image_names_list[day_idx][keep_trials_mask]

    electrode_names_list[day_idx] = electrode_names_list[day_idx][keep_electrodes_mask]
    electrode_roi_list[day_idx] = electrode_roi_list[day_idx][keep_electrodes_mask]

    keep_trials_mask_list.append(keep_trials_mask)
    keep_electrodes_mask_list.append(keep_electrodes_mask)

    n_trials, n_electrodes, n_bins = spikes_list[day_idx].shape
    _, n_pixels, _, n_channels = images_list[day_idx].shape

    n_trials_list.append(n_trials)
    n_electrodes_list.append(n_electrodes)

In [11]:
Conf.data.n_trials_list = n_trials_list
Conf.data.n_electrodes_list = n_electrodes_list
Conf.data.n_bins = len(bin_times)
Conf.data.n_pixels = n_pixels
Conf.data.n_channels = n_channels
Conf.data.n_days = len(day_values)

electrode_names_V1_list = [electrode_names_list[d][electrode_roi_list[d] == 'V1'] for d in range(len(day_values))]
spikes_V1_list = [spikes_list[d][:, electrode_roi_list[d] == 'V1', :] for d in range(len(day_values))]
Conf_V1 = copy.deepcopy(Conf)
Conf_V1.data.n_electrodes_list = [len(x) for x in electrode_names_V1_list]

electrode_names_V4_list = [electrode_names_list[d][electrode_roi_list[d] == 'V4'] for d in range(len(day_values))]
spikes_V4_list = [spikes_list[d][:, electrode_roi_list[d] == 'V4', :] for d in range(len(day_values))]
Conf_V4 = copy.deepcopy(Conf)
Conf_V4.data.n_electrodes_list = [len(x) for x in electrode_names_V4_list]

electrode_names_IT_list = [electrode_names_list[d][electrode_roi_list[d] == 'IT'] for d in range(len(day_values))]
spikes_IT_list = [spikes_list[d][:, electrode_roi_list[d] == 'IT', :] for d in range(len(day_values))]
Conf_IT = copy.deepcopy(Conf)
Conf_IT.data.n_electrodes_list = [len(x) for x in electrode_names_IT_list]

In [12]:
train_dataset_V1_list = []
test_dataset_V1_list = []

train_loader_V1_list = []
test_loader_V1_list = []

Y_mean_V1_list = []
Y_std_V1_list = []

for day_idx in range(len(day_values)):

    roi_mask = electrode_roi_list[day_idx] == "V1"

    train_dataset, test_dataset, Y_mean, Y_std = prepare_dataset(
        Conf_V1,
        images_list[day_idx],
        spikes_list[day_idx][:, roi_mask, :],
        train_idx_list[day_idx],
        test_idx_list[day_idx],
    )

    train_loader, test_loader = prepare_loader(
        Conf_V1,
        train_dataset,
        test_dataset,
    )

    train_dataset_V1_list.append(train_dataset)
    test_dataset_V1_list.append(test_dataset)

    train_loader_V1_list.append(train_loader)
    test_loader_V1_list.append(test_loader)

    Y_mean_V1_list.append(Y_mean)
    Y_std_V1_list.append(Y_std)


train_loader_V1_combined = CombinedLoader(
    {
        str(day_idx): train_loader
        for day_idx, train_loader in enumerate(train_loader_V1_list)
    },
    mode="max_size_cycle",
)

test_loader_V1_combined = CombinedLoader(
    {
        str(day_idx): test_loader
        for day_idx, test_loader in enumerate(test_loader_V1_list)
    },
    mode="max_size_cycle",
)

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


In [13]:
train_dataset_V4_list = []
test_dataset_V4_list = []

train_loader_V4_list = []
test_loader_V4_list = []

Y_mean_V4_list = []
Y_std_V4_list = []

for day_idx in range(len(day_values)):

    roi_mask = electrode_roi_list[day_idx] == "V4"

    train_dataset, test_dataset, Y_mean, Y_std = prepare_dataset(
        Conf_V4,
        images_list[day_idx],
        spikes_list[day_idx][:, roi_mask, :],
        train_idx_list[day_idx],
        test_idx_list[day_idx],
    )

    train_loader, test_loader = prepare_loader(
        Conf_V4,
        train_dataset,
        test_dataset,
    )

    train_dataset_V4_list.append(train_dataset)
    test_dataset_V4_list.append(test_dataset)

    train_loader_V4_list.append(train_loader)
    test_loader_V4_list.append(test_loader)

    Y_mean_V4_list.append(Y_mean)
    Y_std_V4_list.append(Y_std)


train_loader_V4_combined = CombinedLoader(
    {
        str(day_idx): train_loader
        for day_idx, train_loader in enumerate(train_loader_V4_list)
    },
    mode="max_size_cycle",
)

test_loader_V4_combined = CombinedLoader(
    {
        str(day_idx): test_loader
        for day_idx, test_loader in enumerate(test_loader_V4_list)
    },
    mode="max_size_cycle",
)

In [14]:
train_dataset_IT_list = []
test_dataset_IT_list = []

train_loader_IT_list = []
test_loader_IT_list = []

Y_mean_IT_list = []
Y_std_IT_list = []

for day_idx in range(len(day_values)):

    roi_mask = electrode_roi_list[day_idx] == "IT"

    train_dataset, test_dataset, Y_mean, Y_std = prepare_dataset(
        Conf_IT,
        images_list[day_idx],
        spikes_list[day_idx][:, roi_mask, :],
        train_idx_list[day_idx],
        test_idx_list[day_idx],
    )

    train_loader, test_loader = prepare_loader(
        Conf_IT,
        train_dataset,
        test_dataset,
    )

    train_dataset_IT_list.append(train_dataset)
    test_dataset_IT_list.append(test_dataset)

    train_loader_IT_list.append(train_loader)
    test_loader_IT_list.append(test_loader)

    Y_mean_IT_list.append(Y_mean)
    Y_std_IT_list.append(Y_std)


train_loader_IT_combined = CombinedLoader(
    {
        str(day_idx): train_loader
        for day_idx, train_loader in enumerate(train_loader_IT_list)
    },
    mode="max_size_cycle",
)

test_loader_IT_combined = CombinedLoader(
    {
        str(day_idx): test_loader
        for day_idx, test_loader in enumerate(test_loader_IT_list)
    },
    mode="max_size_cycle",
)

## Model Training

In [15]:
cnn_n_layers_list = [4]

trainer_V1_list = []
lit_model_V1_list = []

for cnn_n_layers in cnn_n_layers_list:
    Conf_V1.model_type.cnn_latent.cnn_n_layers = cnn_n_layers

    trainer, lit_model = build_lit_model(Conf_V1, "cnn_latent", True)
    trainer.fit(lit_model, train_loader_V1_combined, test_loader_V1_combined)

    trainer_V1_list.append(trainer)
    lit_model_V1_list.append(lit_model)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Output()

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` 
from an ambiguous collection. The batch size we found is 59. To avoid any miscalculations, use `self.log(..., 
batch_size=batch_size)`.

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` 
from an ambiguous collection. The batch size we found is 64. To avoid any miscalculations, use `self.log(..., 
batch_size=batch_size)`.

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/data.py:79: Trying to infer the `batch_size` 
from an ambiguous collection. The batch size we found is 28. To avoid any miscalculations, use `self.log(..., 
batch_size=batch_size)`.

In [16]:
trainer_IT_list = []
lit_model_IT_list = []

for cnn_n_layers in cnn_n_layers_list:
    Conf_IT.model_type.cnn_latent.cnn_n_layers = cnn_n_layers

    trainer, lit_model = build_lit_model(Conf_IT, "cnn_latent", True)
    trainer.fit(lit_model, train_loader_IT_combined, test_loader_IT_combined)

    trainer_IT_list.append(trainer)
    lit_model_IT_list.append(lit_model)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Output()

In [17]:
trainer_V4_list = []
lit_model_V4_list = []

for cnn_n_layers in cnn_n_layers_list:
    Conf_V4.model_type.cnn_latent.cnn_n_layers = cnn_n_layers

    trainer, lit_model = build_lit_model(Conf_V4, "cnn_latent", True)
    trainer.fit(lit_model, train_loader_V4_combined, test_loader_V4_combined)

    trainer_V4_list.append(trainer)
    lit_model_V4_list.append(lit_model)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Output()

In [18]:
model_path = Path("./models")
model_path.mkdir(parents=True, exist_ok=True)

torch.save(
    [lit_model.state_dict() for lit_model in lit_model_V1_list],
    model_path / "V1-cnn-latent-models.pt",
)

torch.save(
    [lit_model.state_dict() for lit_model in lit_model_V4_list],
    model_path / "V4-cnn-latent-models.pt",
)

torch.save(
    [lit_model.state_dict() for lit_model in lit_model_IT_list],
    model_path / "IT-cnn-latent-models.pt",
)

In [19]:
shutil.make_archive(
    "/kaggle/working/models",
    "zip",
    "/kaggle/working/tvsd-encode",
    "models"
)

'/kaggle/working/models.zip'

In [20]:
for cnn_n_layers, trainer in zip(cnn_n_layers_list, trainer_V1_list):
    plot_training_history(
        trainer=trainer,
        title=f"V1 - CNN Layers {cnn_n_layers}",
        file_name=f"V1-cnn-layers-{cnn_n_layers}-training-history",
        file_path="./plot/model/training_history/V1",
        show=False,
    )

for cnn_n_layers, trainer in zip(cnn_n_layers_list, trainer_IT_list):
    plot_training_history(
        trainer=trainer,
        title=f"IT - CNN Layers {cnn_n_layers}",
        file_name=f"IT-cnn-layers-{cnn_n_layers}-training-history",
        file_path="./plot/model/training_history/IT",
        show=False,
    )

for cnn_n_layers, trainer in zip(cnn_n_layers_list, trainer_V4_list):
    plot_training_history(
        trainer=trainer,
        title=f"V4 - CNN Layers {cnn_n_layers}",
        file_name=f"V4-cnn-layers-{cnn_n_layers}-training-history",
        file_path="./plot/model/training_history/V4",
        show=False,
    )

In [29]:
shutil.make_archive(
    "/kaggle/working/plot",
    "zip",
    "/kaggle/working/tvsd-encode",
    "plot"
)

'/kaggle/working/plot.zip'

## Model Evaluation

In [22]:
model_idx = 0

lit_model_V1 = lit_model_V1_list[model_idx]
lit_model_IT = lit_model_IT_list[model_idx]
lit_model_V4 = lit_model_V4_list[model_idx]

cnn_n_layers = cnn_n_layers_list[model_idx]


Y_test_V1_list, Y_hat_test_V1_list = predict_loader(Conf_V1, lit_model_V1, test_loader_V1_list, Y_mean_V1_list, Y_std_V1_list)
correlation_test_V1, r2_test_V1, mse_test_V1 = compute_metrics(Y_test_V1_list, Y_hat_test_V1_list)

Y_test_V4_list, Y_hat_test_V4_list = predict_loader(Conf_V4, lit_model_V4, test_loader_V4_list, Y_mean_V4_list, Y_std_V4_list)
correlation_test_V4, r2_test_V4, mse_test_V4 = compute_metrics(Y_test_V4_list, Y_hat_test_V4_list)

Y_test_IT_list, Y_hat_test_IT_list = predict_loader(Conf_IT, lit_model_IT, test_loader_IT_list, Y_mean_IT_list, Y_std_IT_list)
correlation_test_IT, r2_test_IT, mse_test_IT = compute_metrics(Y_test_IT_list, Y_hat_test_IT_list)

In [23]:
plot_electrode_metrics(
    correlation_electrode=correlation_test_V1,
    r2_electrode=r2_test_V1,
    mse_electrode=mse_test_V1,
    bin_times=bin_times,
    title=f"V1 - CNN Layers {cnn_n_layers}",
    file_name=f"V1-cnn-layers-{cnn_n_layers}-metrics",
    file_path="./plot/model/roi_metrics",
    show=False,
)

plot_electrode_metrics(
    correlation_electrode=correlation_test_IT,
    r2_electrode=r2_test_IT,
    mse_electrode=mse_test_IT,
    bin_times=bin_times,
    title=f"IT - CNN Layers {cnn_n_layers}",
    file_name=f"IT-cnn-layers-{cnn_n_layers}-metrics",
    file_path="./plot/model/roi_metrics",
    show=False,
)

plot_electrode_metrics(
    correlation_electrode=correlation_test_V4,
    r2_electrode=r2_test_V4,
    mse_electrode=mse_test_V4,
    bin_times=bin_times,
    title=f"V4 - CNN Layers {cnn_n_layers}",
    file_name=f"V4-cnn-layers-{cnn_n_layers}-metrics",
    file_path="./plot/model/roi_metrics",
    show=False,
)

PosixPath('plot/model/roi_metrics')

In [24]:
def compute_test_image_metrics(
    Y_test_list,
    Y_hat_test_list,
    test_idx_list,
    image_names_list,
    bin_times,
):
    test_image_idx = np.unique(
        np.concatenate([
            test_idx[test_idx > 0]
            for test_idx in test_idx_list
        ])
    )

    bin_indices = np.where(
        (bin_times >= 0)
        & (bin_times <= 200)
    )[0]

    correlation_test_image = []
    r2_test_image = []
    mse_test_image = []

    test_image_names = []

    for image_idx in test_image_idx:
        correlation_day_list = []
        r2_day_list = []
        mse_day_list = []

        image_name = None

        for day_idx in range(len(Y_test_list)):
            test_mask = test_idx_list[day_idx] > 0

            test_idx_day = test_idx_list[day_idx][test_mask]
            image_names_day = image_names_list[day_idx][test_mask]

            image_mask = test_idx_day == image_idx

            Y_image = Y_test_list[day_idx][image_mask].mean(axis=0)
            Y_hat_image = Y_hat_test_list[day_idx][image_mask].mean(axis=0)

            n_electrodes = Y_image.shape[1]

            correlation_electrode = np.zeros(n_electrodes)
            r2_electrode = np.zeros(n_electrodes)
            mse_electrode = np.zeros(n_electrodes)

            for electrode_idx in range(n_electrodes):
                y = Y_image[bin_indices, electrode_idx]
                y_hat = Y_hat_image[bin_indices, electrode_idx]

                correlation_electrode[electrode_idx] = compute_correlation(
                    y,
                    y_hat,
                )

                r2_electrode[electrode_idx] = compute_r2(
                    y,
                    y_hat,
                )

                mse_electrode[electrode_idx] = compute_mse(
                    y,
                    y_hat,
                )

            correlation_day_list.append(correlation_electrode)
            r2_day_list.append(r2_electrode)
            mse_day_list.append(mse_electrode)

            if image_name is None:
                image_name = image_names_day[image_mask][0]

        correlation_test_image.append(
            np.concatenate(correlation_day_list)
        )

        r2_test_image.append(
            np.concatenate(r2_day_list)
        )

        mse_test_image.append(
            np.concatenate(mse_day_list)
        )

        test_image_names.append(image_name)

    correlation_test_image = np.stack(
        correlation_test_image,
        axis=0,
    )

    r2_test_image = np.stack(
        r2_test_image,
        axis=0,
    )

    mse_test_image = np.stack(
        mse_test_image,
        axis=0,
    )

    return (
        correlation_test_image,
        r2_test_image,
        mse_test_image,
        test_image_names,
    )

In [25]:
correlation_test_image_V1, r2_test_image_V1, mse_test_image_V1, test_image_names_V1 = compute_test_image_metrics(
    Y_test_V1_list,
    Y_hat_test_V1_list,
    test_idx_list,
    image_names_list,
    bin_times,
)

correlation_test_image_V4, r2_test_image_V4, mse_test_image_V4, test_image_names_V4 = compute_test_image_metrics(
    Y_test_V4_list,
    Y_hat_test_V4_list,
    test_idx_list,
    image_names_list,
    bin_times,
)

correlation_test_image_IT, r2_test_image_IT, mse_test_image_IT, test_image_names_IT = compute_test_image_metrics(
    Y_test_IT_list,
    Y_hat_test_IT_list,
    test_idx_list,
    image_names_list,
    bin_times,
)

In [26]:
plot_image_metrics(
    correlation_image=correlation_test_image_V1,
    r2_image=r2_test_image_V1,
    mse_image=mse_test_image_V1,
    image_names = test_image_names_V1,
    title="V1",
    file_name=f"roi_V1_metrics",
    file_path="./plot/model/image_metrics",
    show=False,
)


plot_image_metrics(
    correlation_image=correlation_test_image_V4,
    r2_image=r2_test_image_V4,
    mse_image=mse_test_image_V4,
    image_names = test_image_names_V4,
    title="V4",
    file_name=f"roi_V4_metrics",
    file_path="./plot/model/image_metrics",
    show=False,
)


plot_image_metrics(
    correlation_image=correlation_test_image_IT,
    r2_image=r2_test_image_IT,
    mse_image=mse_test_image_IT,
    image_names = test_image_names_IT,
    title="IT",
    file_name=f"roi_IT_metrics",
    file_path="./plot/model/image_metrics",
    show=False,
)

PosixPath('plot/model/image_metrics')

In [34]:
image_label = "chipmunk_16n"

image_idx_V1 = test_image_names_V1.index(image_label)
image_idx_V4 = test_image_names_V4.index(image_label)
image_idx_IT = test_image_names_IT.index(image_label)

image_name = image_label.replace("/", "-").replace(".jpg", "")

plot_electrode_metrics_hist(
    correlation_electrode=correlation_test_image_V1[image_idx_V1],
    r2_electrode=r2_test_image_V1[image_idx_V1],
    mse_electrode=mse_test_image_V1[image_idx_V1],
    title=f"{image_label} - V1",
    file_name="V1-metrics-hist",
    file_path=f"./plot/model/image_metrics/{image_name}",
    show=False,
)

plot_electrode_metrics_hist(
    correlation_electrode=correlation_test_image_V4[image_idx_V4],
    r2_electrode=r2_test_image_V4[image_idx_V4],
    mse_electrode=mse_test_image_V4[image_idx_V4],
    title=f"{image_label} - V4",
    file_name="V4-metrics-hist",
    file_path=f"./plot/model/image_metrics/{image_name}",
    show=False,
)

plot_electrode_metrics_hist(
    correlation_electrode=correlation_test_image_IT[image_idx_IT],
    r2_electrode=r2_test_image_IT[image_idx_IT],
    mse_electrode=mse_test_image_IT[image_idx_IT],
    title=f"{image_label} - IT",
    file_name="IT-metrics-hist",
    file_path=f"./plot/model/image_metrics/{image_name}",
    show=False,
)

PosixPath('plot/model/image_metrics/chipmunk_16n')

## Model Interpretation

In [48]:
shutil.make_archive(
    "/kaggle/working/plot",
    "zip",
    "/kaggle/working/tvsd-encode",
    "plot"
)

'/kaggle/working/plot.zip'

In [ ]:
model_idx = 0

lit_model_V1 = lit_model_V1_list[model_idx]
lit_model_V4 = lit_model_V4_list[model_idx]
lit_model_IT = lit_model_IT_list[model_idx]

cnn_n_layers = cnn_n_layers_list[model_idx]

image_label = "pear_13s"

bin_indices_baseline = np.where((bin_times >= -100) & (bin_times <= -50))[0]

roi_data = {
    "V1": {
        "Conf": Conf_V1,
        "lit_model": lit_model_V1,
        "test_dataset_list": test_dataset_V1_list,
        "test_idx_list": test_idx_list,
        "image_names_list": image_names_list,
        "electrode_names_list": electrode_names_V1_list,
        "correlation_test": correlation_test_V1,
        "correlation_test_image": correlation_test_image_V1,
        "test_image_names": test_image_names_V1,
        "n_electrodes_list": Conf_V1.data.n_electrodes_list,
    },
    "V4": {
        "Conf": Conf_V4,
        "lit_model": lit_model_V4,
        "test_dataset_list": test_dataset_V4_list,
        "test_idx_list": test_idx_list,
        "image_names_list": image_names_list,
        "electrode_names_list": electrode_names_V4_list,
        "correlation_test": correlation_test_V4,
        "correlation_test_image": correlation_test_image_V4,
        "test_image_names": test_image_names_V4,
        "n_electrodes_list": Conf_V4.data.n_electrodes_list,
    },
    "IT": {
        "Conf": Conf_IT,
        "lit_model": lit_model_IT,
        "test_dataset_list": test_dataset_IT_list,
        "test_idx_list": test_idx_list,
        "image_names_list": image_names_list,
        "electrode_names_list": electrode_names_IT_list,
        "correlation_test": correlation_test_IT,
        "correlation_test_image": correlation_test_image_IT,
        "test_image_names": test_image_names_IT,
        "n_electrodes_list": Conf_IT.data.n_electrodes_list,
    },
}

for roi_name, roi in roi_data.items():
    Conf_roi = roi["Conf"]
    lit_model = roi["lit_model"].to(Conf_roi.device).eval()

    image_idx = roi["test_image_names"].index(image_label)
    correlation_image = roi["correlation_test_image"][image_idx]

    bin_idx_response = np.argmax(np.nanmean(roi["correlation_test"], axis=0))
    electrode_offsets = np.cumsum(roi["n_electrodes_list"])
    sorted_electrode_idx = np.argsort(correlation_image)[::-1]

    for rank, electrode_global_idx in enumerate(sorted_electrode_idx[1:100]):
        day_idx = np.searchsorted(electrode_offsets, electrode_global_idx, side="right")
        previous_offset = 0 if day_idx == 0 else electrode_offsets[day_idx - 1]
        electrode_idx = electrode_global_idx - previous_offset

        test_mask = roi["test_idx_list"][day_idx] > 0
        image_names_day = roi["image_names_list"][day_idx][test_mask]
        trial_idx = np.where(image_names_day == image_label)[0][0]

        baseline_black = torch.zeros_like(roi["test_dataset_list"][day_idx][trial_idx][0]).unsqueeze(0)

        attribution_maps_baseline = []

        # for bin_idx_baseline in bin_indices_baseline:
        #     attribution_map = compute_attribution_map(
        #         Conf_roi,
        #         lit_model,
        #         roi["test_dataset_list"][day_idx][trial_idx],
        #         day_idx=day_idx,
        #         electrode_idx=electrode_idx,
        #         bin_idx=bin_idx_baseline,
        #         method="occlusion",
        #         baseline=baseline_black,
        #         sliding_window_shapes=(3, 10, 10),
        #         strides=(3, 2, 2),
        #     )

        #     attribution_maps_baseline.append(attribution_map)

        # attribution_maps_baseline = np.stack(attribution_maps_baseline, axis=-1)
        # baseline_mean = attribution_maps_baseline.mean(axis=3)
        # baseline_std = attribution_maps_baseline.std(axis=3)

        baseline_mean = 0
        baseline_std = 1
        
        attribution_map = compute_attribution_map(
            Conf_roi,
            lit_model,
            roi["test_dataset_list"][day_idx][trial_idx],
            day_idx=day_idx,
            electrode_idx=electrode_idx,
            bin_idx=bin_idx_response,
            method="occlusion",
            baseline=baseline_black,
            sliding_window_shapes=(3, 10, 10),
            strides=(3, 2, 2),
        )

        attribution_map_norm = (attribution_map - baseline_mean) / (baseline_std + 1e-8)
        vmax_attribution = np.abs(attribution_map_norm).max()

        electrode_name = roi["electrode_names_list"][day_idx][electrode_idx]

        plot_attribution(
            image=roi["test_dataset_list"][day_idx][trial_idx][0].permute(1, 2, 0).numpy(),
            attribution=attribution_map_norm.transpose(1, 2, 0),
            vmax_attribution=vmax_attribution,
            vmin_attribution=-vmax_attribution,
            title=f"Occlusion Attribution of Electrode {electrode_name} in {roi_name}, Day {day_values[day_idx]}, Rank {rank + 1}, Bin {bin_times[bin_idx_response]}",
            file_name=f"day-{day_values[day_idx]}-electrode-{electrode_name}-rank-{rank + 1}-bin-{bin_idx_response}",
            file_path=f"./plot/model/attribution/occlusion/{roi_name}/cnn_layers_{cnn_n_layers}",
            show=False,
        )